## The first big project - The Digital Twin

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

## Heads up - a change from the videos

In the video, I deploy the twin for free to HuggingFace Spaces. HuggingFace has recently stopped supporting this for free!

There is a free alternative, and I explain it and give instructions later on in this lab.

In [2]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [3]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [4]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [5]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [6]:
push("HEY!!")

Push: HEY!!


In [7]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [8]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [ ]:
import os
import sqlite3

db_folder = "1_foundations"
db_path = os.path.join(db_folder, "qa.db")

os.makedirs(db_folder, exist_ok=True)

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS QA (
    QAId INTEGER PRIMARY KEY AUTOINCREMENT,
    Question TEXT NOT NULL,
    Answer TEXT NOT NULL
)
""")
conn.commit()

def insert_qa(question: str, answer: str):
    cur.execute("INSERT INTO QA (Question, Answer) VALUES (?, ?)", (question, answer))
    conn.commit()
    return cur.lastrowid

def bulk_insert_qas(pairs):
    cur.executemany(
        "INSERT INTO QA (Question, Answer) VALUES (?, ?)",
        [(question, answer) for question, answer in pairs]
    )
    conn.commit()
    return cur.rowcount

def get_answer(question: str):
    cur.execute("SELECT Answer FROM QA WHERE Question = ?", (question,))
    row = cur.fetchone()
    return row[0] if row else None



def delete_all_qas():
    cur.execute("DELETE FROM QA")
    conn.commit()


# Example usage
qa_id = insert_qa("What is SQLite?", "A lightweight embedded SQL database.")
print("Inserted QAId:", qa_id)

print(get_answer("What is SQLite?"))
# print(list_all_qas())

In [72]:
def list_all_qas():
    cur.execute("SELECT QAId, Question, Answer FROM QA")
    return cur.fetchall()

In [59]:
def find_questions_by_text(search_text: str):
    pattern = f"%{search_text}%"
    cur.execute(
        "SELECT Question, Answer FROM QA WHERE Question LIKE ? OR Answer LIKE ?",
        (pattern, pattern)
    )
    return [(row[0], row[1]) for row in cur.fetchall()]

In [9]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [10]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
insert_qa_json ={
  "name": "insert_qa",
  "description": "Inserts a Question and Answer pair into the database and returns the generated row ID.",
  "parameters": {
  "type": "object",
  "properties": {
    "question": {
    "type": "string",
    "description": "The question to be stored."
    },
    "answer": {
    "type": "string",
    "description": "The answer corresponding to the question."
    }
  },
  "required": [
    "question",
    "answer"
  ]
  }
}

In [38]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json},
        {"type": "function", "function": insert_qa_json}]

In [39]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional info about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],


In [13]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

## Using Python built-in globals()

Python has a dictionary that gives us access to all global functions.

Sidenote: for sure when we deploy, we will use this in a more protected way..

In [14]:
globals()["record_unknown_question"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


'OK'

In [15]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [16]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

Inserted QAId: 4
A lightweight embedded SQL database.
[(1, 'What is SQLite?', 'A lightweight embedded SQL database.'), (2, 'What is SQLite?', 'A lightweight embedded SQL database.'), (3, 'What is SQLite?', 'A lightweight embedded SQL database.'), (4, 'What is SQLite?', 'A lightweight embedded SQL database.')]


In [78]:
# delete_all_qas()
# print(list_all_qas())
list_all_qa = list_all_qas()

print(list_all_qa)
# print(find_questions_by_text("study"))
# find_questions_by_text


[(23, 'What is your full name?', 'Ed Donner'), (24, 'What is your profession?', 'Entrepreneur, software engineer, and data scientist'), (25, 'Where are you originally from?', 'London, England'), (26, 'Where did you move in 2000?', 'NYC'), (27, 'What foods do you particularly love?', 'French food'), (28, 'What food do you dislike?', 'Almost all forms of cheese'), (29, 'Which cheeses do you make an exception for?', 'Cream cheese and mozzarella'), (30, 'What are the greatest foods according to you?', 'Cheesecake and pizza'), (31, 'What is your current job title at Nebula.io?', 'Co-Founder & CTO'), (32, 'What company are you currently co-founder and CTO of?', 'Nebula.io'), (33, 'What is your current location?', 'New York, New York, United States'), (34, 'What is your email address?', 'ed.donner@gmail.com'), (35, 'What is your LinkedIn profile URL?', 'www.linkedin.com/in/eddonner'), (36, 'What is your personal website?', 'edwarddonner.com'), (37, 'What are your top skills?', 'CTO, Large Lan

[]


In [45]:
system_prompt_qa = f"""
You are a precise data extraction assistant. Your task is to process the details provided about a person, formulate them into clear Question and Answer (Q&A) pairs, and call a function for each pair.

### Instructions:

1. Analyze the input details regarding the person.
2. Break down the information into atomic, distinct Q&A pairs (e.g., Full Name, Date of Birth, Current Role, Key Skills, Contact Information, etc.).
3. Formulate each question clearly and ensure the answer contains the exact corresponding fact/detail.
4. For EACH Q&A pair generated, execute the function `insert_qa(question, answer)`. Iterate through all pairs in a loop until all details are processed.

### Function Signature:
insert_qa(question: string, answer: string)

### Example Input:
"John Doe is a Senior Software Engineer based in San Francisco with 8 years of experience in Python and cloud architecture. He can be reached at john.doe@email.com."

### Example Tool Calls:
- insert_qa(question="What is the your full name?", answer="John Doe")
- insert_qa(question="What is the your current job title?", answer="Senior Software Engineer")
- insert_qa(question="Where are you located?", answer="San Francisco")
- insert_qa(question="How many years of experience you have?", answer="8 years")
- insert_qa(question="What are your primary technical skills?", answer="Python and cloud architecture")
- insert_qa(question="What is your email address?", answer="john.doe@email.com")

---

### Input Data:
Summary:
{summary}

LinkedIn Profile Details:
{linkedin}
"""

In [23]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up in hindi language.

IMPORTANT:
If you don't know the answer, use your tool to record the question in hindi language, and then tell the user that you don't know. Never make up an answer.
"""


In [46]:

    messages=[
        {"role": "system", "content": system_prompt_qa},
        {"role": "user", "content": ""}
    ]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages,tools=tools,
        tool_choice="auto")
    response_message = response.choices[0].message

    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            if function_name == "insert_qa":
                row_id = insert_qa(
                    question=arguments.get("question"),
                    answer=arguments.get("answer")
                )
                print(f"Inserted row {row_id}: Q: {arguments.get('question')} | A: {arguments.get('answer')}")
            
            elif function_name == "record_user_details_json":
                print(f"Executing record_user_details_json with: {arguments}")
                
            elif function_name == "record_unknown_question_json":
                print(f"Executing record_unknown_question_json with: {arguments}")
        # print(response.choices[0].message.content)

Inserted row 23: Q: What is your full name? | A: Ed Donner
Inserted row 24: Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
Inserted row 25: Q: Where are you originally from? | A: London, England
Inserted row 26: Q: Where did you move in 2000? | A: NYC
Inserted row 27: Q: What foods do you particularly love? | A: French food
Inserted row 28: Q: What food do you dislike? | A: Almost all forms of cheese
Inserted row 29: Q: Which cheeses do you make an exception for? | A: Cream cheese and mozzarella
Inserted row 30: Q: What are the greatest foods according to you? | A: Cheesecake and pizza
Inserted row 31: Q: What is your current job title at Nebula.io? | A: Co-Founder & CTO
Inserted row 32: Q: What company are you currently co-founder and CTO of? | A: Nebula.io
Inserted row 33: Q: What is your current location? | A: New York, New York, United States
Inserted row 34: Q: What is your email address? | A: ed.donner@gmail.com
Inserted row 35: Q: What is yo

In [ ]:
formatted_qa_list = "\n".join([f"- Q: {q} | A: {a}" for qa_id, q, a in list_all_qa])

In [ ]:
print(formatted_qa_list)

- Q: What is your full name? | A: Ed Donner
- Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
- Q: Where are you originally from? | A: London, England
- Q: Where did you move in 2000? | A: NYC
- Q: What foods do you particularly love? | A: French food
- Q: What food do you dislike? | A: Almost all forms of cheese
- Q: Which cheeses do you make an exception for? | A: Cream cheese and mozzarella
- Q: What are the greatest foods according to you? | A: Cheesecake and pizza
- Q: What is your current job title at Nebula.io? | A: Co-Founder & CTO
- Q: What company are you currently co-founder and CTO of? | A: Nebula.io
- Q: What is your current location? | A: New York, New York, United States
- Q: What is your email address? | A: ed.donner@gmail.com
- Q: What is your LinkedIn profile URL? | A: www.linkedin.com/in/eddonner
- Q: What is your personal website? | A: edwarddonner.com
- Q: What are your top skills? | A: CTO, Large Language Models (LLM), PyTorch


In [ ]:
system_prompt_find = f"""You are a precise database lookup assistant. Your task is to check if a user's question can be answered using the provided list of Question and Answer (Q&A) pairs stored in our database.

### Instructions:
1. Analyze the user's input question.
2. Search through the provided Q&A database list to find a matching or semantically equivalent question.
3. If a matching question is found:
   - Acknowledge that the information is in our database.
   - Provide the exact answer corresponding to that question.
4. If a matching question is NOT found in the database:
   - Respond with "I don't know" or state that the information is not available in our database.

### Q&A Database List:
{formatted_qa_list}
"""

In [ ]:
print(system_prompt_find)

You are a precise database lookup assistant. Your task is to check if a user's question can be answered using the provided list of Question and Answer (Q&A) pairs stored in our database.

### Instructions:
1. Analyze the user's input question.
2. Search through the provided Q&A database list to find a matching or semantically equivalent question.
3. If a matching question is found:
   - Acknowledge that the information is in our database.
   - Provide the exact answer corresponding to that question.
4. If a matching question is NOT found in the database:
   - Respond with "I don't know" or state that the information is not available in our database.

### Q&A Database List:
- Q: What is your full name? | A: Ed Donner
- Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
- Q: Where are you originally from? | A: London, England
- Q: Where did you move in 2000? | A: NYC
- Q: What foods do you particularly love? | A: French food
- Q: What food do you dislike

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        print(f"results :", f"{results}\n\n")
        messages.append(message)
        messages.extend(results)
        print(f"messages :", messages);
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
def chat_question(message, history):
    print(f"System Prompt: {system_prompt_find}")   
    # question = message    
    # print(f"History: {json.dumps(history, indent=2)}")  # Print the history for debugging
    messages = [{"role": "system", "content": system_prompt_find}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    # print(f"Answer: {response.choices[0].message.content}")
    answer = response.choices[0].message.content 
    # verified_message = verify(message, response.choices[0].message.content)
    # print(f"Verification Message: {verified_message}") 
    return response.choices[0].message.content

In [93]:
gr.ChatInterface(chat_question).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


System Prompt: You are a precise database lookup assistant. Your task is to check if a user's question can be answered using the provided list of Question and Answer (Q&A) pairs stored in our database.

### Instructions:
1. Analyze the user's input question.
2. Search through the provided Q&A database list to find a matching or semantically equivalent question.
3. If a matching question is found:
   - Acknowledge that the information is in our database.
   - Provide the exact answer corresponding to that question.
4. If a matching question is NOT found in the database:
   - Respond with "I don't know" or state that the information is not available in our database.

### Q&A Database List:
- Q: What is your full name? | A: Ed Donner
- Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
- Q: Where are you originally from? | A: London, England
- Q: Where did you move in 2000? | A: NYC
- Q: What foods do you particularly love? | A: French food
- Q: What food

d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


System Prompt: You are a precise database lookup assistant. Your task is to check if a user's question can be answered using the provided list of Question and Answer (Q&A) pairs stored in our database.

### Instructions:
1. Analyze the user's input question.
2. Search through the provided Q&A database list to find a matching or semantically equivalent question.
3. If a matching question is found:
   - Acknowledge that the information is in our database.
   - Provide the exact answer corresponding to that question.
4. If a matching question is NOT found in the database:
   - Respond with "I don't know" or state that the information is not available in our database.

### Q&A Database List:
- Q: What is your full name? | A: Ed Donner
- Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
- Q: Where are you originally from? | A: London, England
- Q: Where did you move in 2000? | A: NYC
- Q: What foods do you particularly love? | A: French food
- Q: What food

...

d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


System Prompt: You are a precise database lookup assistant. Your task is to check if a user's question can be answered using the provided list of Question and Answer (Q&A) pairs stored in our database.

### Instructions:
1. Analyze the user's input question.
2. Search through the provided Q&A database list to find a matching or semantically equivalent question.
3. If a matching question is found:
   - Acknowledge that the information is in our database.
   - Provide the exact answer corresponding to that question.
4. If a matching question is NOT found in the database:
   - Respond with "I don't know" or state that the information is not available in our database.

### Q&A Database List:
- Q: What is your full name? | A: Ed Donner
- Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
- Q: Where are you originally from? | A: London, England
- Q: Where did you move in 2000? | A: NYC
- Q: What foods do you particularly love? | A: French food
- Q: What food

d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


System Prompt: You are a precise database lookup assistant. Your task is to check if a user's question can be answered using the provided list of Question and Answer (Q&A) pairs stored in our database.

### Instructions:
1. Analyze the user's input question.
2. Search through the provided Q&A database list to find a matching or semantically equivalent question.
3. If a matching question is found:
   - Acknowledge that the information is in our database.
   - Provide the exact answer corresponding to that question.
4. If a matching question is NOT found in the database:
   - Respond with "I don't know" or state that the information is not available in our database.

### Q&A Database List:
- Q: What is your full name? | A: Ed Donner
- Q: What is your profession? | A: Entrepreneur, software engineer, and data scientist
- Q: Where are you originally from? | A: London, England
- Q: Where did you move in 2000? | A: NYC
- Q: What foods do you particularly love? | A: French food
- Q: What food

d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Turning into Python modules

I've turned the code in the lab into python modules; that's a great practice to do after you've completed experiments in the Notebook.

You could put all the code above into 1 python script. But it's nicer to organize the code into different modules for different concerns, and that's what I've done:

`context.py` loads in the static data and constructs the System Prompt

`tools.py` contains all the code to manage and call tools, with their associated json

`app.py` contains the Gradio app and OpenAI call.

`styles.py` contains styles to apply to Gradio and this was entirely written by Claude Code!

You could have a stab at doing this yourself, then compare with my versions.

Then to try it out, open a terminal in Cursor:

`cd 1_foundations`  
`cd twin`  
`uv run app.py`

# STOP THE PRESS! Heads up...

As of 9-July-2026, HuggingFace has suddenly stopped allowing Gradio Apps to be deployed for free on HuggingFace Spaces.

This is quite a nasty surprise!

I expect they might reverse this decision. In the meantime, here's a free alternative: using Render.

You'll find complete instructions in [the file RENDER_INSTRUCTIONS in this directory](RENDER_INSTRUCTIONS.md)

If you don't mind paying for HuggingFace, the original instructions are below.

And also, here are instructions on my digital twin, which runs at very low cost on fly.io:  
https://edwarddonner.com/avatar

With my twin, not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the exercise in Day 4, and add other Agentic patterns.<br/>
            • Some students have added Telegram integration so that you can chat live with people on your site, along with your twin!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your resume of the future) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>